In [17]:
pairs = [
    {
        "original": """
        Heat olive oil in a large skillet over medium heat. Add garlic and cook for 1–2 minutes until fragrant.
        Add tomatoes and simmer for 10 minutes. Boil pasta until al dente, then mix with sauce and serve.
        """,
        "simplified": """
        Heat oil in a pan.
        Add garlic and cook until fragrant.
        Add tomatoes and simmer.
        Cook pasta until al dente.
        Mix with sauce and serve.
        """
    }
]

In [18]:
# Step 1 Standardize the text before scoring
# Before computing anything, make sure both texts are in a comparable form.
# Handles things like:
    # remove extra spaces
    # make sure text is a string
    # handle empty or missing output safely

def standardize_text(text):
    if not isinstance(text, str):
        return ""
    return ' '.join(text.split())

In [19]:
# Step 2: Write one function just for semantic similarity
# This function:
    # take original text
    # take simplified text
    # return one similarity score


# Sentence Transformers
# specifically a pretrained embedding model such as:
# all-MiniLM-L6-v2

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_semantic_similarity(list):
    original = list['original']
    simplified = list['simplified']
    
    # Standardize the texts
    original_std = standardize_text(original)
    simplified_std = standardize_text(simplified)
    
    # Compute embeddings
    embedding_original = model.encode(original_std, convert_to_tensor=True)
    embedding_simplified = model.encode(simplified_std, convert_to_tensor=True)
    
    # Compute cosine similarity
    similarity_score = util.cos_sim(embedding_original, embedding_simplified).item()
    
    return similarity_score


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# Step 3: Write one function just for compression ratio
# This function:
    # compare the length of simplified text to original text
    # Internally, it should:
    # count words in original
    # count words in simplified
    # divide simplified length by original length
    # return the ratio

# avoid crashing, if original text length is zero,

def compute_compression_ratio(list):
    original = list['original']
    simplified = list['simplified']
    
    # Standardize the texts
    original_std = standardize_text(original)
    simplified_std = standardize_text(simplified)
    
    # Count words
    original_word_count = len(original_std.split())
    simplified_word_count = len(simplified_std.split())
    
    # Avoid division by zero
    if original_word_count == 0:
        return 0.0
    
    # Compute compression ratio
    compression_ratio = simplified_word_count / original_word_count
    
    return compression_ratio

In [21]:
# Step 4: Write one function just for readability
# This function:
    # score how easy the simplified text is to read

# compute readability for the original text too, so you can compare before vs after.
#comout loss, to show the redability redunction?

# using:
# textstat

# It computes a readability metric such as:
# Flesch-Kincaid Grade Level

import textstat
def compute_readability(list):
    original = list['original']
    simplified = list['simplified']
    
    # Standardize the texts
    original_std = standardize_text(original)
    simplified_std = standardize_text(simplified)
    
    # Compute readability scores
    readability_original = textstat.flesch_kincaid_grade(original_std)
    readability_simplified = textstat.flesch_kincaid_grade(simplified_std)
    
    return readability_original, readability_simplified

In [25]:
def evaluate_simplification(list):
    similarity = compute_semantic_similarity(list)
    compression = compute_compression_ratio(list)
    readability_original, readability_simplified = compute_readability(list)

    readability_improvement = 1 - (readability_simplified / readability_original)
    overall_score = (similarity + compression + readability_improvement) / 3

    if overall_score >= 0.80:
        rating = "Very Strong Simplification"
    elif overall_score >= 0.65:
        rating = "Good"
    elif overall_score >= 0.50:
        rating = "Moderate"
    else:
        rating = "Weak"

    print("=" * 40)
    print("       SIMPLIFICATION EVALUATION")
    print("=" * 40)
    print(f"  Semantic Similarity:      {similarity:.4f}")
    print(f"  Compression Ratio:        {compression:.4f}")
    print(f"  Readability (Original):   {readability_original:.4f}")
    print(f"  Readability (Simplified): {readability_simplified:.4f}")
    print(f"  Readability Improvement:  {readability_improvement:.4f}")
    print("-" * 40)
    print(f"  Overall Score:            {overall_score:.4f}")
    print(f"  Rating:                   {rating}")
    print("=" * 40)

    return {
        "similarity": similarity,
        "compression": compression,
        "readability_original": readability_original,
        "readability_simplified": readability_simplified,
        "readability_improvement": readability_improvement,
        "overall_score": overall_score,
        "rating": rating
    }

In [26]:
import ssl                                                                                                     
import nltk
import os                                                                                                      
                  
ssl._create_default_https_context = ssl._create_unverified_context                                             
   
nltk_path = os.path.expanduser('~/nltk_data')                                                                  
result = nltk.download('cmudict', download_dir=nltk_path)
                                                                                                                 
                                                               
os.makedirs(nltk_path, exist_ok=True)
                                                                                                                              
print(result)   
print(nltk.data.find('corpora/cmudict'))  

True
/Users/nkemokweye/nltk_data/corpora/cmudict


[nltk_data] Downloading package cmudict to
[nltk_data]     /Users/nkemokweye/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


In [27]:
evaluate_simplification(pairs[0]);

       SIMPLIFICATION EVALUATION
  Semantic Similarity:      0.8439
  Compression Ratio:        0.6757
  Readability (Original):   4.6013
  Readability (Simplified): 1.9360
  Readability Improvement:  0.5792
----------------------------------------
  Overall Score:            0.6996
  Rating:                   Good
